This notebook includes runs related to unmixing by using:

1. A very simple MLP, of various different sizes. These sizes are:
    - [32,32]
    - [64^3]
    - [128^4]
    - [256^4]
2. Of varying learning rates fine tuned for the given model.
3. With dropout, optimized using AdamW.
4. A loss of simple MSE, with a small additional terms for sum of residuals.
5. With a training loop that loops over the entire training dataset at each epoch, with a max epochs of 80.
6. Data separated into 1500 training samples, 100 static validation samples, and 123 static test samples.
7. Trained using a GPU.
8. Statistically normalized datasets.
9. Activations of LeakyReLU throughout the model, following BatchNorm, with the final activation layer of ReLU+L1norm.
10. Models initialized using Kaiming (He), tuned for LeakyReLUl leak constant of 0.2.


### Imports, data, model initialization, and various other setups

In [21]:
# Set up autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
# Sys Imports
import sys
from pathlib import Path

github_root = Path.cwd().resolve().parents[2]
sys.path.append(str(github_root))

In [23]:
# Self-def imports

from src.dl.data import get_data
from src.dl.plotting import plot_preds, plot_avg_losses, plot_metrics, plot_train_losses, pareto_plot
from src.dl.models.mlp import MLP
from src.dl.train import train_model

In [24]:
# Package imports
import torch
import torch.nn as nn

In [25]:
# Get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [27]:
# Get data
save_path = rf"{github_root}\data\simpler_data_rwc.csv"

dataloaders = get_data(save_path=save_path, spec_range=[900, 1700])
epoch = 80
n_tb_epoch = 189

In [28]:
# Define model configs

models_list = [
    {'hidden_dim': [32, 32]},
    {'hidden_dim': [64, 64, 64]},
    {'hidden_dim': [128, 128, 128, 128]},
    {'hidden_dim': [256, 256, 256, 256]}
]

lr_list = [1e-3, 5e-4, 1e-4, 5e-5]

### Model train loop

In [29]:
cfg_dict = models_list[0]

model = MLP(**cfg_dict).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr_list[0], weight_decay=1e-2)

lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epoch*n_tb_epoch, eta_min=1e-6)

loss = nn.HuberLoss()

In [30]:
model_losses_step, model_losses_average, model_metrics_epoch, model_metrics_total = train_model(
    model=model,
    loss_fn=loss,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    configured_data=dataloaders,
    n_epoch=epoch,
    n_tb_epoch=n_tb_epoch,
    device=device,
    dtype=torch.float32,
    model_save='a.pth',
    test_save='b'
)


Epoch 1/80, Step 1/189, Train Loss: 0.122907
Epoch 1/80, Step 2/189, Train Loss: 0.076217
Epoch 1/80, Step 3/189, Train Loss: 0.126952
Epoch 1/80, Step 4/189, Train Loss: 0.052104
Epoch 1/80, Step 5/189, Train Loss: 0.105090
Epoch 1/80, Step 6/189, Train Loss: 0.118584
Epoch 1/80, Step 7/189, Train Loss: 0.091158
Epoch 1/80, Step 8/189, Train Loss: 0.089669
Epoch 1/80, Step 9/189, Train Loss: 0.053132
Epoch 1/80, Step 10/189, Train Loss: 0.069694
Epoch 1/80, Step 11/189, Train Loss: 0.093381
Epoch 1/80, Step 12/189, Train Loss: 0.086742
Epoch 1/80, Step 13/189, Train Loss: 0.123453
Epoch 1/80, Step 14/189, Train Loss: 0.016965
Epoch 1/80, Step 15/189, Train Loss: 0.073976
Epoch 1/80, Step 16/189, Train Loss: 0.034489
Epoch 1/80, Step 17/189, Train Loss: 0.140905
Epoch 1/80, Step 18/189, Train Loss: 0.045301
Epoch 1/80, Step 19/189, Train Loss: 0.035795
Epoch 1/80, Step 20/189, Train Loss: 0.056944
Epoch 1/80, Step 21/189, Train Loss: 0.062534
Epoch 1/80, Step 22/189, Train Loss: 0.0398

In [31]:
print(model_metrics_total)

[[3.90700000e+03 7.42400000e+03 7.05282483e-03 7.60970354e-01
  8.38084340e-01 5.98689437e-01 8.46137285e-01 8.52883756e-02
  7.07129538e-02 1.12089105e-01 7.30630681e-02]]
